# CUDA Benchmark for SimSearch
This notebook runs the CLIP zero-shot image classification benchmark on a cloud GPU.

In [ ]:
# 1. Check if GPU is available
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU found. Go to Runtime -> Change runtime type -> Select GPU.")

In [ ]:
# 2. Install dependencies
!pip install transformers

In [ ]:
# 3. Create images directory and populate with test images
import os
from PIL import Image, ImageDraw
import urllib.request

IMAGE_DIR = 'images'
os.makedirs(IMAGE_DIR, exist_ok=True)

# Try downloading sample images, fallback to synthetic if it fails
samples = {
    'dog.jpg': 'https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg',
    'car.jpg': 'https://raw.githubusercontent.com/pytorch/hub/master/images/car.jpg'
}

for name, url in samples.items():
    path = os.path.join(IMAGE_DIR, name)
    if not os.path.exists(path):
        print(f"Attempting to download {name}...")
        try:
            urllib.request.urlretrieve(url, path)
            print(f"Downloaded {name}")
        except Exception as e:
            print(f"Download failed for {name}: {e}. Creating synthetic image.")
            # Create a synthetic image
            img = Image.new('RGB', (640, 480), color = (73, 109, 137))
            d = ImageDraw.Draw(img)
            d.text((20, 20), f"Synthetic {name}", fill=(255, 255, 0))
            img.save(path)

print(f"Images in '{IMAGE_DIR}':", os.listdir(IMAGE_DIR))

In [ ]:
# 4. Run the Benchmark
import time
from transformers import pipeline, AutoProcessor, AutoModelForZeroShotImageClassification
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch32"

print(f"Loading model on target device: {DEVICE}...")

model = AutoModelForZeroShotImageClassification.from_pretrained(model_id)
processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "zero-shot-image-classification", 
    model=model, 
    feature_extractor=processor.image_processor, 
    tokenizer=processor.tokenizer,
    device=0 if DEVICE == "cuda" else -1,
    model_kwargs={"torch_dtype": torch.float16 if DEVICE == "cuda" else torch.float32}
)

candidate_labels = [
    "a photo of a cat", "Daughter", "a photo of a dog", "a photo of a car",
    "woman", "man", "girl", "boy", "child", "brother", "sister", "family",
    "mom", "dad", "group photo", "couple", "friends", "cruise", "garden",
    "men", "women"
]

valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
image_files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(valid_extensions)]

if not image_files:
    print(f"No images found in '{IMAGE_DIR}'.")
else:
    print(f"Found {len(image_files)} image(s). Warming up GPU...")
    if DEVICE == "cuda":
        image_warmup = Image.open(os.path.join(IMAGE_DIR, image_files[0])).convert("RGB")
        _ = pipe(image_warmup, candidate_labels=candidate_labels)
        torch.cuda.synchronize()
    
    print("Starting benchmark...")
    start_time = time.time()
    
    for filename in image_files:
        filepath = os.path.join(IMAGE_DIR, filename)
        print(f"--- Processing: {filename} ---")
        try:
            image = Image.open(filepath).convert("RGB")
            results = pipe(image, candidate_labels=candidate_labels)
            for result in results[:3]: # Print top 3
                print(f"  -> Label: {result['label']} - Score: {result['score']:.4f}")
        except Exception as e:
            print(f"  -> Failed to process {filename}: {e}")
            
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        
    end_time = time.time()
    processing_time = end_time - start_time
    
    print("=========================================")
    print(f"      BENCHMARKING RESULTS ({DEVICE.upper()})      ")
    print("=========================================")
    print(f"Total images processed : {len(image_files)}")
    print(f"Total processing time  : {processing_time:.2f} seconds")
    print(f"Average time per image : {processing_time / len(image_files):.2f} seconds")
    print("=========================================")